**Policy Iteration**

In this exercise, we are going to implement Policy Iteration, an iterative method to obtain the optimal policy, in the next MDP:

![alt text](two_state_mdp.png "Title")

Let us start with the imports. We use only numpy and matplotlib in this exercise.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

Now, let us define the parameters of the problem. We have an MDP with two states, and two actions. The rewards, discount factor, and transition probabilities are given in the figure above. We also know the optimal policy beforehand from previous exercises:

In [2]:
gamma = 0.9
R = np.array([[-1, 0.6, 0.5, -0.9]]).T
P = np.array([[0.8, 0.2], [0.2, 0.8], [0.3, 0.7], [0.9, 0.1]])
pi_opt = np.array([[0, 1, 0, 0], [0, 0, 1, 0]])

First, we obtain the optimal value function $v^{\pi^* }$ and the optimal action-value function $q^{\pi^*}$ using the fixed-point Bellman equations, in order to assess the accuracy of our implementation:
* $v^{\pi} = \left( I - \gamma \mathcal{P}^{\pi} \right)^{-1} \mathcal{R}^{\pi}$
* $q^{\pi} = \left( I - \gamma \mathcal{P} \Pi \right)^{-1} \mathcal{R}$

In [3]:
v_opt = (np.linalg.inv(np.eye(pi_opt.shape[0]) - gamma * pi_opt @ P) @ pi_opt @ R).flatten()
q_opt = (np.linalg.inv(np.eye(P.shape[0]) - gamma * P @ pi_opt) @ R).flatten()
with np.printoptions(precision=2, suppress=True):
    print(f"Optimal Policy = {pi_opt.flatten()}")
    print(f"v^* = {v_opt}")
    print(f"q^* = {q_opt}")

Optimal Policy = [0 1 0 0 0 0 1 0]
v^* = [5.34 5.25]
q^* = [3.79 5.34 5.25 3.9 ]


Now, we are going to implement Policy Iteration, an iterative method, for the state value function, following the algorithm seen in the slides:

In [4]:
n_states = 2
n_actions = 2
pi_pe = [np.zeros((n_states, n_states * n_actions))]  # Initial policy
pi_pe[-1][0,0] = 1
pi_pe[-1][1,3] = 1

v_pi_pe = [np.zeros((n_states, 1))]  # Initialize the value function to 0
threshold = 1e-3  # Variation change for convergence check in PE
theta = False  # Indicator of PI convergence
i = 0  # PI iterations

while not theta:
    # First thing: PE loop (Policy Evaluation)
    delta = 1.0
    v_k = v_pi_pe[-1].copy()
    current_pi = pi_pe[-1]
    
    while delta > threshold:
        v_next = (current_pi @ R) + gamma * (current_pi @ P) @ v_k
        delta = np.max(np.abs(v_next - v_k))
        v_k = v_next
        
    v_pi_pe.append(v_k)

    # Now, PI loop: Policy Improvement
    # Calculate Q-values from the newly evaluated V
    q_values = R + gamma * P @ v_k
    
    # Construct a new greedy policy
    new_pi = np.zeros((n_states, n_states * n_actions))
    new_pi[0, np.argmax(q_values[0:2])] = 1      # Best action for State 0
    new_pi[1, 2 + np.argmax(q_values[2:4])] = 1  # Best action for State 1
    
    # Check if the policy has stopped changing
    if np.array_equal(new_pi, current_pi):
        theta = True
    else:
        pi_pe.append(new_pi)
        
    i += 1

print('PI converged after ', i, ' iterations')
with np.printoptions(precision=2, suppress=True):  # Print the values obtained
    print(f"Policy optimal theory = {pi_opt.flatten()}")
    print(f"Policy optimal PI = {pi_pe[-1].flatten()}")
    print(f"v^* theory = {v_opt}")
    print(f"v^* PI = {v_pi_pe[-1].flatten()}")

PI converged after  2  iterations
Policy optimal theory = [0 1 0 0 0 0 1 0]
Policy optimal PI = [0. 1. 0. 0. 0. 0. 1. 0.]
v^* theory = [5.34 5.25]
v^* PI = [5.33 5.24]


And we are going to repeat the procedure, but using Policy Iteration on the state-action value function, following the algorithm seen in the slides:

In [7]:
n_states = 2
n_actions = 2
pi_pe = [np.zeros((n_states, n_states * n_actions))]  
pi_pe[-1][0,0] = 1
pi_pe[-1][1,3] = 1

q_pi_pe = [np.zeros((n_states * n_actions, 1))]  
threshold = 1e-3  
theta = False  
i = 0  

while not theta:
    # First thing: PE loop (Policy Evaluation)
    delta = 1.0
    q_k = q_pi_pe[-1].copy()
    current_pi = pi_pe[-1]
    
    while delta > threshold:
        q_next = R + gamma * P @ (current_pi @ q_k)
        delta = np.max(np.abs(q_next - q_k))
        q_k = q_next
        
    q_pi_pe.append(q_k)

    # Now, PI loop: Policy Improvement directly from Q-values
    new_pi = np.zeros((n_states, n_states * n_actions))
    new_pi[0, np.argmax(q_k[0:2])] = 1      # Best action for State 0
    new_pi[1, 2 + np.argmax(q_k[2:4])] = 1  # Best action for State 1
    
    if np.array_equal(new_pi, current_pi):
        theta = True
    else:
        pi_pe.append(new_pi)
        
    i += 1

print('PI converged after ', i, ' iterations')
with np.printoptions(precision=2, suppress=True):  
    print(f"Policy optimal theory = {pi_opt.flatten()}")
    print(f"Policy optimal PI = {pi_pe[-1].flatten()}")
    print(f"q^* theory = {q_opt}")
    print(f"q^* PI = {q_pi_pe[-1].flatten()}")

PI converged after  2  iterations
Policy optimal theory = [0 1 0 0 0 0 1 0]
Policy optimal PI = [0. 1. 0. 0. 0. 0. 1. 0.]
q^* theory = [3.79 5.34 5.25 3.9 ]
q^* PI = [3.78 5.33 5.24 3.89]
